In [37]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

In [41]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7
)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [42]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [43]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [44]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [45]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [46]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

/home/surendra/Desktop/LangGraph/venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/surendra/Desktop/LangGraph/venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'My wife told me to stop impersonating a flamingo. \n\nI had to put my foot down. \n\n...Which is also what I do when the pizza delivery guy is at the door.',
   'extras': {'signature': 'Eo4BCosBAWkUfRMnUkga/ABc+q/128yWixgtG8+MJtJwGohOOOJpRy33HQKSTia4b5lOmB7vEazroL5ZsyuPjkELa7eErB9mcPbuG5eDFzwE8vqXDTVMHMOEoDYsyUi3jweshcBX/toiCSqFw5BzbuLpDE4VussMN/8+duciFRQsvG6H1euQuhqD1quwUKLXZA=='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a **double meaning (pun)** based on the physical behavior of a flamingo.\n\nHere is the breakdown of why it\'s funny:\n\n1. **The Setup:** Flamingos are famous for standing on one leg. The wife tells her husband to stop impersonating (copying) one.\n2. **The Punchline:** He replies, *"I had to put my foot down."* \n   * **Literal meaning:** In the context of the flamingo, "putting your foot down" means lowering your raised leg back to the ground (which a flamingo does when it stops s